# Extract & Label Chess Piece Glyphs from PDF

**Workflow:**
1. Load previous classifier (optional) to pre-filter candidates
2. Extract candidate glyphs from PDF
3. Manually label ambiguous ones
4. Train new classifier
5. Download zip with labeled glyphs + classifier

## Step 1 — Mount Google Drive and load PDF

In [ ]:
from google.colab import drive
import os

drive.mount('/content/gdrive')

In [ ]:
PDF_PATH = '/content/gdrive/MyDrive/chess_book.pdf'

if not os.path.exists(PDF_PATH):
    print(f'❌ PDF not found: {PDF_PATH}')
else:
    size_mb = os.path.getsize(PDF_PATH) / (1024*1024)
    print(f'✅ PDF loaded: {PDF_PATH}  ({size_mb:.1f} MB)')

## Step 2 — Install dependencies

In [ ]:
!apt-get install -y poppler-utils
!pip install -q pdfplumber pdf2image pillow scikit-learn scikit-image opencv-python

## Step 3 — Configuration

In [ ]:
import pdfplumber
from pdf2image import convert_from_path
from PIL import Image
import os
from pathlib import Path
import pickle
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from skimage import filters, transform
import zipfile

START_PAGE = 1
END_PAGE = 20
GLYPH_DPI = 150
IMG_SIZE = 32
CONFIDENCE_THRESHOLD = 0.7
CLASSIFIER_PREFILTER = True   # set False when classifier is not trained on current book style

GLYPHS_DIR = './glyphs_results/glyphs'
WORK_DIR = './glyphs_results'
PIECE_CLASSES = ['K', 'Q', 'R', 'B', 'N']

Path(WORK_DIR).mkdir(exist_ok=True)
Path(GLYPHS_DIR).mkdir(exist_ok=True)
for piece in PIECE_CLASSES:
    Path(f'{GLYPHS_DIR}/{piece}').mkdir(exist_ok=True)
import cv2
from skimage import filters, transform

def extract_image_features(img):
    if img.width < 5 or img.height < 5:
        return None
    img_resized = transform.resize(np.array(img), (IMG_SIZE, IMG_SIZE), anti_aliasing=True)
    features = []
    features.append(np.mean(img_resized))
    features.append(np.std(img_resized))
    features.append(img.width / max(img.height, 1))
    edges = filters.sobel(img_resized)
    features.append(np.mean(edges))
    features.append(np.mean(np.sum(img_resized, axis=0)))
    features.append(np.mean(np.sum(img_resized, axis=1)))
    return np.array(features)

def render_page(pdf_path, page_num, dpi=150):
    images = convert_from_path(pdf_path, first_page=page_num+1, last_page=page_num+1, dpi=dpi)
    return images[0] if images else None

print(f'Output directory: {WORK_DIR}/')

## Step 3.5 — Load previous classifier (optional)

In [ ]:
from google.colab import files
import glob

classifier = None
book_templates = {}
previous_labeled_count = {piece: 0 for piece in PIECE_CLASSES}

print('Upload a previous classifier zip (optional, press Skip if none):')

try:
    uploaded = files.upload()
    print(f'Uploaded files: {list(uploaded.keys())}')

    for filename in uploaded.keys():
        if not filename.endswith('.zip'):
            continue
        print(f'Extracting {filename}...')
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall('./previous')

        # Load classifier
        classifier = None
        for fmt, loader in [
            ('./previous/glyphs_results/classifier.keras',
             lambda p: __import__('tensorflow').keras.models.load_model(p)),
            ('./previous/glyphs_results/classifier.pkl',
             lambda p: __import__('pickle').load(open(p, 'rb'))),
        ]:
            if os.path.exists(fmt):
                classifier = loader(fmt)
                print(f'Classifier loaded from {fmt}')
                break
        if classifier is None:
            print('No classifier found in zip.')

        # Load and merge book_templates (append styles, don't replace)
        templates_path = './previous/glyphs_results/book_templates.pkl'
        if os.path.exists(templates_path):
            with open(templates_path, 'rb') as f:
                loaded_templates = pickle.load(f)
            for piece, tmpl_list in loaded_templates.items():
                if piece not in book_templates:
                    book_templates[piece] = []
                book_templates[piece].extend(tmpl_list)
            summary = {p: len(v) for p, v in book_templates.items()}
            print('Book templates loaded: ' + ', '.join(f'{p}x{n}' for p, n in sorted(summary.items())))
        else:
            print('No book_templates.pkl — crop templates manually in Step 4.6.')

        # Copy labeled glyphs into current session so retraining accumulates all history
        copied = {piece: 0 for piece in PIECE_CLASSES}
        for piece in PIECE_CLASSES:
            src_dir = f'./previous/glyphs_results/glyphs/{piece}'
            dst_dir = f'{GLYPHS_DIR}/{piece}'
            if not os.path.exists(src_dir):
                continue
            existing = len(glob.glob(f'{dst_dir}/*.png'))
            for i, src_file in enumerate(sorted(glob.glob(f'{src_dir}/*.png'))):
                dst_file = f'{dst_dir}/{existing + i + 1:04d}.png'
                import shutil as _shutil
                _shutil.copy2(src_file, dst_file)
                copied[piece] += 1
        total_copied = sum(copied.values())
        if total_copied:
            summary_str = '  '.join(f'{p}:{copied[p]}' for p in PIECE_CLASSES if copied[p])
            print(f'Labeled glyphs copied into session: {total_copied}  ({summary_str})')
        else:
            print('No labeled glyphs found in zip.')

except Exception as e:
    print(f'Error loading zip: {e}')
    import traceback
    traceback.print_exc()

## Step 4.6 — Crop piece templates from the book itself

Since the PDF is a scanned document, the chess pieces are embedded in the page images.
Use the sliders below to crop one clean example of each piece type directly from a page you know contains chess symbols.
These **book templates** will replace the sprite templates for matching.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import matplotlib.patches as patches

if 'book_templates' not in globals():
    book_templates = {}  # {piece: [template_img, ...]}

with pdfplumber.open(PDF_PATH) as _pdf:
    _total_pages = len(_pdf.pages)

page_sel = widgets.BoundedIntText(value=11, min=1, max=_total_pages,
                                  description='Page:', layout=widgets.Layout(width='140px'))
load_btn = widgets.Button(description='Load page', button_style='primary')

def coord_row(label, lo, hi, val):
    slider = widgets.IntSlider(min=lo, max=hi, value=val, description=label,
                               continuous_update=False,
                               layout=widgets.Layout(width='350px'))
    editor = widgets.BoundedIntText(min=lo, max=hi, value=val,
                                    layout=widgets.Layout(width='90px'))
    widgets.link((slider, 'value'), (editor, 'value'))
    return slider, editor, widgets.HBox([slider, editor])

x1_s, x1_t, x1_row = coord_row('x1', 0, 800, 0)
y1_s, y1_t, y1_row = coord_row('y1', 0, 800, 0)
x2_s, x2_t, x2_row = coord_row('x2', 1, 800, 60)
y2_s, y2_t, y2_row = coord_row('y2', 1, 800, 60)

status = widgets.Label('Saved so far: none')
out    = widgets.Output()
_current_page_image = [None]

def update_status():
    summary = {p: len(v) for p, v in book_templates.items()}
    status.value = 'Templates: ' + ', '.join(f'{p}x{n}' for p, n in sorted(summary.items()))

def update_coord_limits(w, h):
    for s, t, mx in [(x1_s, x1_t, w-1), (y1_s, y1_t, h-1),
                     (x2_s, x2_t, w),   (y2_s, y2_t, h)]:
        s.max = mx
        t.max = mx

def load_page(_=None):
    with out:
        clear_output(wait=True)
        print(f'Rendering page {page_sel.value}...')
    img = render_page(PDF_PATH, page_sel.value - 1, dpi=GLYPH_DPI)
    _current_page_image[0] = img
    arr = np.array(img)
    h, w = arr.shape[:2]
    update_coord_limits(w, h)
    x1_s.value, y1_s.value = 0, 0
    x2_s.value, y2_s.value = min(60, w), min(60, h)
    update_preview()

def update_preview(*_):
    img = _current_page_image[0]
    if img is None:
        return
    x1, y1 = x1_s.value, y1_s.value
    x2, y2 = max(x2_s.value, x1+1), max(y2_s.value, y1+1)
    arr = np.array(img)
    with out:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        axes[0].imshow(arr)
        axes[0].add_patch(patches.Rectangle(
            (x1, y1), x2-x1, y2-y1,
            linewidth=2, edgecolor='red', facecolor='none'))
        axes[0].set_title(f'Page {page_sel.value}  --  red box = crop region')
        axes[1].imshow(arr[y1:y2, x1:x2])
        axes[1].set_title(f'Crop preview  {x2-x1}x{y2-y1} px')
        plt.tight_layout()
        plt.show()

def make_save(piece):
    def _save(_):
        img = _current_page_image[0]
        if img is None:
            status.value = 'Load a page first'
            return
        x1, y1 = x1_s.value, y1_s.value
        x2, y2 = max(x2_s.value, x1+1), max(y2_s.value, y1+1)
        crop_gray = cv2.cvtColor(
            np.array(img.crop((x1, y1, x2, y2))), cv2.COLOR_RGB2GRAY)
        if piece not in book_templates:
            book_templates[piece] = []
        book_templates[piece].append(crop_gray)  # append, never replace
        update_status()
    return _save

load_btn.on_click(load_page)
for s in [x1_s, y1_s, x2_s, y2_s]:
    s.observe(update_preview, names='value')

save_btns = [widgets.Button(description=p, button_style='info',
                            layout=widgets.Layout(width='60px'))
             for p in ['K', 'Q', 'R', 'B', 'N', 'P']]
for btn, piece in zip(save_btns, ['K', 'Q', 'R', 'B', 'N', 'P']):
    btn.on_click(make_save(piece))

display(widgets.HBox([page_sel, load_btn]))
display(x1_row)
display(y1_row)
display(x2_row)
display(y2_row)
display(widgets.HBox([widgets.Label("Save crop as ->")  ] + save_btns))
display(status)
display(out)
update_status()
load_page()

In [ ]:
import sys

glyph_words = []

book_templates = globals().get('book_templates', {})
active_templates = book_templates if book_templates else {}
template_source = 'book' if book_templates else 'none'

if not active_templates:
    print('No book templates found — run Step 4.6 first to crop piece templates.')
else:
    total_tmpl = sum(len(v) for v in active_templates.values())
    print(f'Matching {len(active_templates)} piece types, {total_tmpl} templates total ({template_source})...', flush=True)
    with pdfplumber.open(PDF_PATH) as pdf:
        pdf_page_count = len(pdf.pages)
        end_page = min(END_PAGE, pdf_page_count)

        for page_idx in range(START_PAGE - 1, end_page):
            page_num = page_idx + 1
            print(f'  Page {page_num:3d}: rendering...', end='', flush=True)

            page_image = render_page(PDF_PATH, page_idx, dpi=GLYPH_DPI)
            if page_image is None:
                print(' skipped', flush=True)
                continue

            page_cv = cv2.cvtColor(np.array(page_image), cv2.COLOR_RGB2GRAY)
            page_counts = {piece: 0 for piece in active_templates}

            for piece_name, template_list in active_templates.items():
                for template in template_list:
                    if template.shape[0] == 0 or template.shape[1] == 0:
                        continue
                    result = cv2.matchTemplate(page_cv, template, cv2.TM_CCOEFF_NORMED)
                    locs = np.where(result >= 0.7)
                    th, tw = template.shape
                    for y, x in zip(locs[0], locs[1]):
                        crop = page_image.crop((x, y, x+tw, y+th))
                        features = extract_image_features(crop)
                        if features is None:
                            continue
                        is_duplicate = any(
                            abs(x - e['bbox'][0]) < tw/2 and abs(y - e['bbox'][1]) < th/2
                            for e in glyph_words
                        )
                        if not is_duplicate:
                            glyph_words.append({
                                'page': page_num,
                                'text': piece_name,
                                'bbox': (x, y, x+tw, y+th),
                                'crop': crop,
                                'features': features,
                                'source': 'template',
                            })
                            page_counts[piece_name] += 1
                            total = len(glyph_words)
                            print(f'\r  Page {page_num:3d}: +{piece_name} at ({x},{y})  [total: {total}]', end='', flush=True)

            counts_str = '  '.join(f'{p}:{page_counts[p]}' for p in active_templates)
            print(f'\r  Page {page_num:3d}: {counts_str}', flush=True)

    print(f'\nFound {len(glyph_words)} chess piece candidates total', flush=True)
    print(f'Classifier available: {classifier is not None}')

    if classifier is not None:
        for word in glyph_words:
            word['confidence'] = float(np.max(classifier.predict(np.array([word['features']], dtype='float32'), verbose=0)[0]))

        # Confidence breakdown per piece before filtering
        print(f'\nConfidence breakdown (threshold = {CONFIDENCE_THRESHOLD}):')
        print(f'  {"Piece":6}  {"Total":>6}  {">=thr":>6}  {"<thr":>6}  {"min":>6}  {"avg":>6}  {"max":>6}')
        all_pieces = sorted(set(w['text'] for w in glyph_words))
        for piece in all_pieces:
            confs = [w['confidence'] for w in glyph_words if w['text'] == piece]
            above = sum(1 for c in confs if c >= CONFIDENCE_THRESHOLD)
            below = len(confs) - above
            print(f'  {piece:6}  {len(confs):>6}  {above:>6}  {below:>6}  {min(confs):>6.2f}  {sum(confs)/len(confs):>6.2f}  {max(confs):>6.2f}')

        if CLASSIFIER_PREFILTER:
            before = len(glyph_words)
            glyph_words = [w for w in glyph_words if w['confidence'] >= CONFIDENCE_THRESHOLD]
            print(f'\nClassifier pre-filtered: {before} -> {len(glyph_words)} candidates')
        else:
            print(f'\nClassifier confidence annotated, pre-filter disabled ({len(glyph_words)} candidates kept)')
    else:
        print('No classifier available; showing all candidates')

## Step 5 — Label candidates

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

current_idx = 0
saved_count = {piece: 0 for piece in PIECE_CLASSES}
skipped = 0
discarded = 0

def show_glyph(idx):
    global current_idx, saved_count, skipped, discarded
    if idx >= len(glyph_words):
        clear_output()
        print(f'✅ Labeling complete!')
        print(f'Saved:')
        for piece in PIECE_CLASSES:
            print(f'  {piece}: {saved_count[piece]}')
        print(f'  Skipped: {skipped}')
        print(f'  Discarded: {discarded}')
        return
    current_idx = idx
    word_info = glyph_words[idx]
    crop = word_info['crop']
    clear_output()
    conf_text = f" (confidence: {word_info.get('confidence', 0):.2%})" if 'confidence' in word_info else ""
    print(f'Glyph {idx + 1}/{len(glyph_words)} — P{word_info["page"]}: "{word_info["text"]}"{conf_text}')
    print()
    display(crop)
    print()
    def save_glyph(piece):
        count = saved_count[piece]
        filename = f'{GLYPHS_DIR}/{piece}/{count + 1:04d}.png'
        crop.save(filename)
        saved_count[piece] += 1
        show_glyph(idx + 1)
    def skip():
        global skipped
        skipped += 1
        show_glyph(idx + 1)
    def discard():
        global discarded
        discarded += 1
        show_glyph(idx + 1)
    buttons = [
        widgets.Button(description='K (King)', button_style='info'),
        widgets.Button(description='Q (Queen)', button_style='info'),
        widgets.Button(description='R (Rook)', button_style='info'),
        widgets.Button(description='B (Bishop)', button_style='info'),
        widgets.Button(description='N (Knight)', button_style='info'),
        widgets.Button(description='Skip', button_style='warning'),
        widgets.Button(description='❌ Discard', button_style='danger'),
    ]
    buttons[0].on_click(lambda _: save_glyph('K'))
    buttons[1].on_click(lambda _: save_glyph('Q'))
    buttons[2].on_click(lambda _: save_glyph('R'))
    buttons[3].on_click(lambda _: save_glyph('B'))
    buttons[4].on_click(lambda _: save_glyph('N'))
    buttons[5].on_click(lambda _: skip())
    buttons[6].on_click(lambda _: discard())
    display(widgets.HBox(buttons))

if len(glyph_words) > 0:
    show_glyph(0)
else:
    print('❌ No candidates to label')

## Step 6 — Train classifier

In [ ]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers

X_train = []
y_train = []
for piece_idx, piece in enumerate(PIECE_CLASSES):
    path = f'{GLYPHS_DIR}/{piece}'
    if os.path.exists(path):
        for img_file in sorted(os.listdir(path)):
            if img_file.endswith('.png'):
                img = Image.open(f'{path}/{img_file}').convert('L')
                features = extract_image_features(img)
                if features is not None:
                    X_train.append(features)
                    y_train.append(piece_idx)

if len(X_train) > 10:
    X_train = np.array(X_train, dtype='float32')
    y_train = np.array(y_train)

    model = keras.Sequential([
        layers.Input(shape=(6,)),
        layers.Dense(64, activation='relu'),
        layers.Dense(64, activation='relu'),
        layers.Dense(len(PIECE_CLASSES), activation='softmax'),
    ])
    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    model.fit(X_train, y_train, epochs=100, batch_size=16, verbose=0)

    _, acc = model.evaluate(X_train, y_train, verbose=0)
    model.save(f'{WORK_DIR}/classifier.keras')
    classifier = model

    counts = {p: y_train.tolist().count(i) for i, p in enumerate(PIECE_CLASSES)}
    print(f'Trained on {len(X_train)} samples  accuracy: {acc:.1%}')
    print('  ' + '  '.join(f'{p}:{counts[p]}' for p in PIECE_CLASSES))
else:
    print('Not enough labeled samples (need >= 10)')

## Step 6.5 — Export TFLite model

In [ ]:
import tensorflow as tf

converter = tf.lite.TFLiteConverter.from_keras_model(classifier)
tflite_model = converter.convert()

tflite_path = f'{WORK_DIR}/classifier.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

print(f'TFLite model saved: {tflite_path}  ({len(tflite_model)/1024:.1f} KB)')

## Step 7 — Export zip with classifier

In [ ]:
import shutil

# Save book_templates alongside classifier
book_templates_to_save = globals().get('book_templates', {})
if book_templates_to_save:
    templates_path = f'{WORK_DIR}/book_templates.pkl'
    with open(templates_path, 'wb') as f:
        pickle.dump(book_templates_to_save, f)
    summary = {p: len(v) for p, v in book_templates_to_save.items()}
    print('Book templates saved: ' + ', '.join(f'{p}x{n}' for p, n in sorted(summary.items())))
else:
    print('No book templates to save.')

zip_filename = 'chess_glyphs_classifier.zip'
shutil.make_archive('chess_glyphs_classifier', 'zip', '.', WORK_DIR)

final_counts = {}
total_glyphs = 0
for piece in PIECE_CLASSES:
    path = f'{GLYPHS_DIR}/{piece}'
    if os.path.exists(path):
        count = len([f for f in os.listdir(path) if f.endswith('.png')])
        final_counts[piece] = count
        total_glyphs += count

print(f'Export complete: {zip_filename}')
print(f'  glyphs_results/')
print(f'  ├── classifier.keras')
    print(f'  ├── classifier.tflite')
print(f'  ├── book_templates.pkl')
print(f'  └── glyphs/')
for piece in PIECE_CLASSES:
    count = final_counts.get(piece, 0)
    print(f'      ├── {piece}/ ({count} images)')
print(f'Total: {total_glyphs} labeled glyphs')

from google.colab import files
files.download(zip_filename)